In [ ]:
import os
import numpy as np
import torch
from PIL import Image, ImageFile
from TransNetV2.inference_pytorch.transnetv2_pytorch import TransNetV2 as TransNetV2_pytorch
from TransNetV2.inference.transnetv2 import TransNetV2 as TransNetV2_utils
import ffmpeg
import cv2
import imagehash
from concurrent.futures import ThreadPoolExecutor, as_completed
from transformers import BlipProcessor, BlipForConditionalGeneration, pipeline
from sentence_transformers import SentenceTransformer
import faiss
import sqlite3
import gc

# Handle truncated images
ImageFile.LOAD_TRUNCATED_IMAGES = True

db_path = "video_processing_results.db"
device = "cuda" if torch.cuda.is_available() else "cpu"

# Initialize models
device = torch.device(device)
pipe = pipeline("image-to-text", model="Salesforce/blip-image-captioning-large")
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-large")
model_blip = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-large").to(device)

sentence_model = SentenceTransformer("all-MiniLM-L6-v2")
index = faiss.IndexFlatL2(384)  # 384 is the dimension of embeddings from 'all-MiniLM-L6-v2'

model_transnet = TransNetV2_pytorch().to(device)
model_transnet.load_state_dict(torch.load("transnetv2-pytorch-weights.pth", map_location=device))
model_transnet.eval()

def create_database(db_path: str) -> sqlite3.Connection:
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS videos (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            video_id INTEGER UNIQUE,
            video_path TEXT UNIQUE
        )
    ''')
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS scenes (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            video_id INTEGER,
            scene_start TEXT,
            scene_end TEXT,
            keyframe_url TEXT,
            caption TEXT,
            FOREIGN KEY (video_id) REFERENCES videos(id)
        )
    ''')
    cursor.execute('CREATE INDEX IF NOT EXISTS idx_video_id ON videos (video_id)')
    cursor.execute('CREATE INDEX IF NOT EXISTS idx_scene_timestamps ON scenes (scene_start, scene_end)')
    cursor.execute('CREATE INDEX IF NOT EXISTS idx_caption ON scenes (caption)')
    conn.commit()
    return conn

def insert_video(conn: sqlite3.Connection, video_id: int, video_path: str) -> int:
    cursor = conn.cursor()
    cursor.execute('INSERT OR IGNORE INTO videos (video_id, video_path) VALUES (?, ?)', (video_id, video_path))
    conn.commit()
    return cursor.lastrowid

def insert_scene(conn: sqlite3.Connection, video_id: int, scene_start: str, scene_end: str, keyframe_url: str, caption: str) -> int:
    cursor = conn.cursor()
    cursor.execute('''
        INSERT INTO scenes (video_id, scene_start, scene_end, keyframe_url, caption)
        VALUES (?, ?, ?, ?, ?)
    ''', (video_id, scene_start, scene_end, keyframe_url, caption))
    conn.commit()
    return cursor.lastrowid

def generate_caption(image_path: str) -> str:
    with Image.open(image_path).convert('RGB') as raw_image:
        inputs = processor(raw_image, return_tensors="pt").to(device)
        out = model_blip.generate(**inputs, max_new_tokens=50)
        caption = processor.decode(out[0], skip_special_tokens=True)
    return caption

def frames_to_timecode(frames: int, fps: float) -> str:
    total_seconds = frames / fps
    hours = int(total_seconds // 3600)
    minutes = int((total_seconds % 3600) // 60)
    seconds = int(total_seconds % 60)
    milliseconds = int((total_seconds - int(total_seconds)) * 1000)
    return f"{hours:02}:{minutes:02}:{seconds:02}.{milliseconds:03}"

def get_video_fps(video_path: str) -> float:
    probe = ffmpeg.probe(video_path)
    video_info = next(stream for stream in probe['streams'] if stream['codec_type'] == 'video')
    fps_str = video_info['r_frame_rate']
    num, denom = map(int, fps_str.split('/'))
    fps = num / denom
    return fps

def extract_frames_as_jpg(video_path: str, start_time: str, end_time: str, width: int = 1920, height: int = 1080) -> list[str]:
    out, _ = (
        ffmpeg
        .input(video_path, ss=start_time, to=end_time)
        .output('pipe:', format='rawvideo', pix_fmt='rgb24', s=f'{width}x{height}')
        .run(capture_stdout=True, capture_stderr=True)
    )
    frame_count = len(out) // (width * height * 3)
    if frame_count == 0:
        raise ValueError("No frames extracted.")
    
    frames = np.frombuffer(out, np.uint8).reshape([frame_count, height, width, 3])
    frame_filenames = []

    for idx, frame in enumerate(frames):
        frame_filename = f"frame_{idx:04d}.jpg"
        image = Image.fromarray(frame)
        image.save(frame_filename)
        image.close()  
        frame_filenames.append(frame_filename)

    return frame_filenames

def select_representative_frame(images: list[str], similarity_threshold: float = 0.75) -> str:
    unique_images = []
    hashes = []

    for image_path in images:
        with Image.open(image_path) as img:
            phash = imagehash.phash(img)
            if all(phash - h > similarity_threshold for h in hashes):
                hashes.append(phash)
                unique_images.append(image_path)

    if not unique_images:
        return images[0]

    # Custom saliency detection to find the most representative frame
    def compute_saliency(image_path: str) -> float:
        img = cv2.imread(image_path)
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        fft = np.fft.fft2(gray)
        magnitude_spectrum = 20 * np.log(np.abs(np.fft.fftshift(fft)) + 1e-10)  # Add small constant to avoid log(0)
        saliency_score = np.mean(magnitude_spectrum)
        return saliency_score

    best_image_path = unique_images[0]
    max_saliency = -1

    with ThreadPoolExecutor(max_workers=4) as executor:  # Limit the number of threads
        future_to_image = {executor.submit(compute_saliency, image_path): image_path for image_path in unique_images}
        for future in as_completed(future_to_image):
            image_path = future_to_image[future]
            
            saliency_score = future.result()
            if saliency_score > max_saliency:
                max_saliency = saliency_score
                best_image_path = image_path
            
    return best_image_path

def process_video(video_path: str, video_id: int, parent_folder: str, conn: sqlite3.Connection):

    insert_video(conn, video_id, video_path)
    fps = get_video_fps(video_path)

    # Initialize the TransNetV2 utils model
    transnet_utils = TransNetV2_utils()
    video_frames, single_frame_predictions, all_frame_predictions = transnet_utils.predict_video(video_path)
    scenes = transnet_utils.predictions_to_scenes(single_frame_predictions)

    predictions = (single_frame_predictions + all_frame_predictions) / 2
    scenes = transnet_utils.predictions_to_scenes(predictions)

    # Check if the model returned the whole video as a single scene or no scenes at all
    video_duration = len(video_frames) / fps
    max_scene_length_frames = 60 * fps

    if scenes is None or len(scenes) == 0 or (len(scenes) == 1 and scenes[0][0] == 0 and scenes[0][1] >= len(video_frames) - 1):
        scenes = [(i * max_scene_length_frames, min((i + 1) * max_scene_length_frames, len(video_frames) - 1))
                    for i in range(int(video_duration // 60))]
        if video_duration % 60 != 0:
            scenes.append((int(video_duration // 60) * max_scene_length_frames, len(video_frames) - 1))

    # Visualize predictions
    visualization_image = transnet_utils.visualize_predictions(video_frames, [single_frame_predictions, all_frame_predictions])
    visualization_image.save(os.path.join(parent_folder, 'shot_boundaries_visualization.png'))
    visualization_image.close()  

    for (start, end) in scenes:
        # If the scene is longer than max_scene_length, only consider the first 60 seconds
        if (end - start) > max_scene_length_frames:
            end = start + max_scene_length_frames

        start_timecode = frames_to_timecode(start, fps)
        end_timecode = frames_to_timecode(end, fps)

        frame_filenames = extract_frames_as_jpg(video_path, start_timecode, end_timecode)
        best_image_path = select_representative_frame(frame_filenames)

        scene_folder = os.path.join(parent_folder, f"{start_timecode}-{end_timecode}".replace(':', '_'))
        os.makedirs(scene_folder, exist_ok=True)
        output_image_path = os.path.join(scene_folder, f'keyframe.jpg')
        with Image.open(best_image_path) as best_image:
            best_image.save(output_image_path)

        caption = generate_caption(output_image_path)
        insert_scene(conn, video_id, start_timecode, end_timecode, output_image_path, caption)

        for frame_filename in frame_filenames:
            os.remove(frame_filename)

        gc.collect()

In [ ]:
conn = create_database(db_path)
video_dir = r'E:\V3C1-100'
list= [4,5,6,7,8,9,99]
for video_id in list:
    video_path = os.path.join(video_dir, f'001{video_id:02}', f'001{video_id:02}.mp4')
    if os.path.exists(video_path):
        parent_folder = f"ID{video_id:02}"
        os.makedirs(parent_folder, exist_ok=True)
        process_video(video_path, video_id, parent_folder, conn)

conn.close()